# 03 - Logistic scorecard

I use the WoE columns from Notebook 2 to fit the logistic model on the training loans. Then I check the three splits, look at validation deciles, and turn the coefficients into score points. The scale is 600 at 50:1 good-to-bad odds, with 20 points per doubling.


In [1]:
# Load the WoE splits and binning setup from Notebook 2.
import os
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, log_loss

LOCAL_DATA = Path(r"E:\CRM Projects\Data")
if "POCKET_CRM_DATA" not in os.environ and LOCAL_DATA.is_dir():
    os.environ["POCKET_CRM_DATA"] = str(LOCAL_DATA)
DATA = Path(os.environ.get("POCKET_CRM_DATA", "./data")).expanduser().resolve()
OUT = DATA / "pocket1_model"

pipe = joblib.load(OUT / "feature_pipeline.joblib")
TARGET, features = pipe["target"], pipe["selected"]
splits = {name: pd.read_parquet(OUT / f"woe_{name}.parquet")
          for name in ("train", "validation", "test")}


In [2]:
# Fit on the training split.
train = splits["train"]
model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
# scikit-learn 1.8 warns about C when penalty=None.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message=r"Setting penalty=None will ignore",
        category=UserWarning,
    )
    model.fit(train[features], train[TARGET].astype(int))

coef = pd.Series(model.coef_[0], index=features)
print("Intercept:", model.intercept_[0])
print("Coefficients:\n", coef.round(4))
# Positive WoE is safer here; flag coefficients that run the other way.
if (coef >= 0).any():
    print("Review nonnegative coefficients:", coef[coef >= 0].to_dict())


Intercept: -6.203128518175588
Coefficients:
 fico                     -0.8172
ltv                      -0.6294
original_interest_rate   -0.5289
original_loan_term       -0.2518
dti                      -0.4430
seller_name              -0.3730
original_upb             -0.2757
dtype: float64


In [3]:
# Check ranking and PD error on each split.
reports, scored = [], []
for name, loans in splits.items():
    y = loans[TARGET].astype(int)
    pd_hat = model.predict_proba(loans[features])[:, 1]
    fpr, tpr, _ = roc_curve(y, pd_hat)
    auc = roc_auc_score(y, pd_hat)

    reports.append(dict(split=name, AUC=auc, KS=(tpr - fpr).max(), Gini=2*auc-1,
                        Brier=brier_score_loss(y, pd_hat), log_loss=log_loss(y, pd_hat),
                        observed=y.mean(), predicted=pd_hat.mean()))
    scored.append(loans[["loan_id", TARGET]].assign(split=name, PD=pd_hat,
                   log_odds=model.decision_function(loans[features])))
metrics = pd.DataFrame(reports).set_index("split")
scored = pd.concat(scored, ignore_index=True)
print(metrics.round(4))


               AUC      KS    Gini   Brier  log_loss  observed  predicted
split                                                                    
train       0.8129  0.5045  0.6258  0.0020    0.0129     0.002      0.002
validation  0.8236  0.5258  0.6472  0.0020    0.0129     0.002      0.002
test        0.7790  0.4299  0.5580  0.0019    0.0131     0.002      0.002


In [4]:
# Validation loans, from highest to lowest predicted risk.
v = scored.loc[scored["split"].eq("validation")].copy()
v["decile"] = pd.qcut(-v["PD"], 10, labels=False, duplicates="drop") + 1
calibration = v.groupby("decile").agg(loans=(TARGET, "size"),
    defaults=(TARGET, "sum"), observed_PD=(TARGET, "mean"), predicted_PD=("PD", "mean"))
print("Decile 1 has the highest predicted risk:\n", calibration.round(4))


Decile 1 has the highest predicted risk:
         loans  defaults  observed_PD  predicted_PD
decile                                            
1        5733        51       0.0089        0.0088
2        5756        26       0.0045        0.0039
3        5691        14       0.0025        0.0025
4        5724        11       0.0019        0.0017
5        5718         4       0.0007        0.0012
6        5726         2       0.0003        0.0008
7        5724         3       0.0005        0.0006
8        5730         0       0.0000        0.0004
9        5729         1       0.0002        0.0002
10       5707         1       0.0002        0.0001


In [5]:
# Higher score means lower risk: 600 at 50:1, 20 PDO.
factor = 20 / np.log(2)
offset = 600 - factor * np.log(50)
scored["score"] = offset - factor * scored["log_odds"]
base_points = offset - factor * model.intercept_[0]

# Put each WoE bin on the points scale.
bins = pd.read_csv(OUT / "binning_tables.csv")
points = bins.loc[bins["feature"].isin(features),
                  ["feature", "Bin", "WoE_used"]].copy()
points["points"] = -factor * points["feature"].map(coef) * points["WoE_used"]
points = pd.concat([pd.DataFrame([{"feature":"(base)", "Bin":"(all)",
                                   "WoE_used":np.nan, "points":base_points}]),
                    points], ignore_index=True)
print("First scorecard rows:\n", points.head())


First scorecard rows:
   feature               Bin  WoE_used      points
0  (base)             (all)       NaN  666.107331
1    fico    (-inf, 641.50) -1.277169  -30.116592
2    fico  [641.50, 665.50) -0.861173  -20.307106
3    fico  [665.50, 714.50) -0.225438   -5.315995
4    fico  [714.50, 760.50)  0.561085   13.230804


In [6]:
# Save the model and scores for Notebook 4.
OUT.mkdir(exist_ok=True)
champion = {**pipe, "model":model, "factor":factor, "offset":offset}
joblib.dump(champion, OUT / "woe_logistic_champion.joblib")
scored.drop(columns="log_odds").to_parquet(OUT / "scored_loans.parquet", index=False)
metrics.to_csv(OUT / "metrics.csv")
calibration.to_csv(OUT / "validation_calibration.csv")
points.to_csv(OUT / "scorecard_points.csv", index=False)
print("Saved logistic champion and validation results to", OUT)


Saved logistic champion and validation results to E:\CRM Projects\Data\pocket1_model
